In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [18]:
device = "cpu"

In [7]:
class FlightDataset(Dataset):
    def __init__(self, sequences_cat, sequences_cont, labels):
        self.sequences_cat = torch.tensor(sequences_cat,  dtype=torch.long)
        self.sequences_cont = torch.tensor(sequences_cont, dtype=torch.float32)
        self.labels = torch.tensor(labels,         dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.sequences_cat[idx],
            self.sequences_cont[idx],
            self.labels[idx]
        )

In [9]:
class FlightDelayRNN(nn.Module):
    def __init__(
        self,
        n_origins,    
        n_dests,      
        embed_dim=16, 
        n_cont_features
    ):
        super(FlightDelayRNN, self).__init__()

        self.origin_embed  = nn.Embedding(n_origins,  embed_dim)
        self.dest_embed    = nn.Embedding(n_dests,    embed_dim)
        self.carrier_embed = nn.Embedding(12, embed_dim) #12 carriers
        self.daysofweek_embed = nn.Embedding(7, embed_dim) #7 days of week

        self.cont_bn = nn.BatchNorm1d(n_cont_features) #batch normalization

        lstm_input_size = (embed_dim * 4) + n_cont_features #4 categorical features need embedding

        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=24,
            batch_first=True,
        )

        self.dense = nn.Linear(24,1)

    def forward(self, x_cat, x_cont):
        """
        x_cat:  (batch_size, seq_len, 3)         — [origin, dest, carrier]
        x_cont: (batch_size, seq_len, n_cont)    — continuous features
        """
        batch_size, seq_len, _ = x_cat.shape

        #embedding categorical features
        origin_emb = self.origin_embed(x_cat[:, :, 0])   # (batch, seq_len, embed_dim)
        dest_emb = self.dest_embed(x_cat[:, :, 1])     # (batch, seq_len, embed_dim)
        carrier_emb = self.carrier_embed(x_cat[:, :, 2])  # (batch, seq_len, embed_dim)
        daysofweek_emb = self.daysofweek_embed(x_cat[:,:,3]) # (batch, seq_len, embed_dim)

        #normalizing continuous features
        # BatchNorm1d expects (batch, features) — reshape, normalize, reshape back
        x_cont_flat = x_cont.view(-1, x_cont.shape[-1])         # (batch*seq_len, n_cont)
        x_cont_norm = self.cont_bn(x_cont_flat)
        x_cont_norm = x_cont_norm.view(batch_size, seq_len, -1) # (batch, seq_len, n_cont)

        # --- Concatenate all features ---
        x = torch.cat([origin_emb, dest_emb, carrier_emb, daysofweek_emb, x_cont_norm], dim=-1)
        # x shape: (batch, seq_len, embed_dim*3 + n_cont)

        output, (h_n, c_n) = self.lstm(x)

        return torch.flatten(self.dense(output[:,-1])) #not sure if this is right

In [11]:
def train_model(model, train_loader, val_loader):  
    criterion = torch.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    best_val_loss = float('inf')

    #20 epochs for now
    for epoch in range(20):
        model.train()
        train_losses = []

        #training
        for x_cat, x_cont, labels in train_loader:
            x_cat = x_cat.to(device)
            x_cont = x_cont.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            predictions = model(x_cat, x_cont)
            loss = criterion(predictions, labels)
            loss.backward()

            # Gradient clipping — important for RNNs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_losses.append(loss.item())

        
        model.eval()
        val_losses = []

        #validation
        with torch.no_grad():
            for x_cat, x_cont, labels in val_loader:
                x_cat = x_cat.to(device)
                x_cont = x_cont.to(device)
                labels = labels.to(device)

                predictions = model(x_cat, x_cont)
                loss = criterion(predictions, labels)
                val_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)
        scheduler.step(avg_val_loss)

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        print(f'Epoch {epoch+1:3d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f}')

    print('Training complete. Best model saved.')

In [ ]:
df = pd.read_csv('C:\\Users\\natha\\Documents\\combined_weather_dec2020\\2020_12_with_weather.csv')

In [50]:
cat_features = ['DayOfWeek', 'Dest', 'Origin', 'Reporting_Airline']
cont_features = ['Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10', 
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10', 'DepDelay']

In [70]:
import datetime
import airportsdata
import zoneinfo

In [64]:
iata_airports = airportsdata.load('IATA')

In [76]:
def convert_date(year, month, day, hhmm, airport):
    dt = datetime.strptime(str(hhmm).zfill(4), '%H%M')
    dt = dt.replace(year = year, month=month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
    return dt.astimezone(timezone.utc)

In [78]:
def build_sequences(data, sequence_length):
    data['UTC_CRSDepTime'] = data.apply(lambda row: convert_date(row['Year'], row['Month'], row['Day'], row['CRSDepTime'], row['Origin']))
    

In [ ]:
df['origin_enc'] = OneHotEncoder(drop='first', handle_unknown='ignore').fit_transform(df['Origin'])
df['dest_enc'] = OneHotEncoder(drop='first', handle_unknown='ignore').fit_transform(df['Dest'])
df['carrier_enc'] = OneHotEncoder(drop='first', handle_unknown='ignore').fit_transform(df['Reporting_Airline'])
df['daysofweek_enc'] = OneHotEncoder(drop='first').fit_transform(df['DayOfWeek'])

X_cat, X_cont, y = build_sequences(df, sequence_length=5)

n = len(y)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_cat_train,  X_cat_val,  X_cat_test = X_cat[:train_end],  X_cat[train_end:val_end],  X_cat[val_end:]
X_cont_train, X_cont_val, X_cont_test = X_cont[:train_end], X_cont[train_end:val_end], X_cont[val_end:]
y_train, y_val, y_test = y[:train_end], y[train_end:val_end],      y[val_end:]

train_dataset = FlightDataset(X_cat_train, X_cont_train, y_train)
val_dataset = FlightDataset(X_cat_val,   X_cont_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=256, shuffle=False)

model = FlightDelayRNN(
    n_origins = df['origin_enc'].nunique(),
    n_dests = df['dest_enc'].nunique(),
).to(device)

train_model(model, train_loader, val_loader)